# Практическая работа № 3. Свёрточные нейронные сети
## Классификация цветов
**Дисциплина:** Прикладной искусственный интеллект.

**Выполнил(а):** ФИО, ИТМО ID.

Авторы практикума: Ким Станислав Александрович, Евстафьев Олег Александрович.

Обязательны три модели: исходная CNN, предобученная архитектура, собственная сеть. Этот ноутбук запускает одну выбранную модель за проход; повторите раздел обучения для всех трёх, сохраняя общий протокол.

**Первое знакомство с изображениями.** Откройте одну фотографию и найдите её класс. Тензор — массив пикселей; пакет — несколько изображений, обрабатываемых вместе. Сеть выдаёт пять оценок, по одной на класс.

Сначала проверьте форму входа и выхода. До обучения исходной большой CNN прочитайте расчёт памяти в пособии. Предобученный извлекатель превращает изображение в признаки; новая голова переводит их в пять оценок классов.

Выполняйте ячейки сверху вниз. После изменения исходных данных или перед сдачей перезапустите среду и выполните всё заново. Для запуска скачайте весь проект: учебные модули находятся в общей папке `code`, а данные — в папке `data` рядом с этим ноутбуком.

## 1. Среда
Сохраните пять каталогов изображений в `data`. Источник исходного задания: [Flowers Recognition](https://www.kaggle.com/datasets/alxmamaev/flowers-recognition). Проверьте версии PyTorch/torchvision и ускоритель. Исходная CNN содержит 207 175 365 параметров; заранее оцените память.

In [ ]:
from pathlib import Path
import sys
import os
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (CURRENT_DIR, *CURRENT_DIR.parents)
     if (p / "code").is_dir() and (p / "labs").is_dir()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the complete practicum project with code/ and labs/")
LAB_DIR = PROJECT_ROOT / "labs" / "03_cnn"
sys.path.insert(0, str(PROJECT_ROOT / "code"))
os.chdir(LAB_DIR)
print("Working directory:", LAB_DIR)
import json
import torch
from torch import nn
import torchvision
import pandas as pd
import matplotlib.pyplot as plt
from lab3_utils import prepare_flowers, seed_everything, original_cnn
from lab3_utils import transfer_resnet18, fit_flowers, plot_learning
from lab3_utils import parameter_counts, denormalize
seed_everything(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__, torchvision.__version__, device)


## 2. Данные и единое разбиение
1000 — ориентир размера валидации из исходника; группировка точных дубликатов может изменить фактическое число. Для маленького учебного поднабора задайте другое значение до эксперимента. Сохраните один и тот же объект data для всех трёх моделей.

In [ ]:
data = prepare_flowers("data", validation_target=1000,
                       seed=0, image_size=224, augment=False)
print(data["class_to_idx"])
print(data["config"])
display(data["inventory"].groupby(["split", "target"]).size())


**Ваши наблюдения:** размеры изображений, частоты классов, дубликаты, смысл нормализации. Объясните, почему аугментации валидации должны отличаться от обучающих.

## 3. Архитектуры
Проверьте размерности и число параметров заданной CNN без выделения памяти под её большие веса. Meta-устройство используется только для расчёта формы, обучать на нём нельзя.

In [ ]:
with torch.device("meta"):
    reference = original_cnn()
    shape = reference(torch.empty(1, 3, 224, 224)).shape
print("Original CNN:", parameter_counts(reference), "output:", shape)


### Собственная архитектура
Ниже дан компактный пример. **Внесите собственное обоснованное структурное изменение до запуска student.** Объясните гипотезу, формы тензоров и число параметров. Не представляйте неизменённый пример как свою разработку.

In [ ]:
def make_student_model():
    # Modify this architecture and explain the change in the report.
    model = nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 5)
    )
    model.lab_description = {"architecture": "student_model", "pretrained": False}
    return model


## 4. Запуск одной модели
Выберите `original`, `transfer` или `student`. Перед сменой модели сохраните графики предыдущей; при необходимости освободите память. Всегда создавайте новую модель. В transfer загружаются ImageNet-веса; `pretrained=False` не выполняет требуемый перенос обучения. При нехватке памяти пересмотрите пакет для всего основного сравнения, сохранив новую версию протокола.

In [ ]:
ARCHITECTURE = "transfer"
SEED, EPOCHS, BATCH_SIZE, LEARNING_RATE = 0, 10, 32, 1e-3
seed_everything(SEED)
if ARCHITECTURE == "original":
    model = original_cnn()
elif ARCHITECTURE == "transfer":
    model = transfer_resnet18(pretrained=True)
elif ARCHITECTURE == "student":
    model = make_student_model()
else:
    raise ValueError("Choose original, transfer, or student")
print(parameter_counts(model))
OUTPUT_DIR = Path("runs") / f"lab3_{ARCHITECTURE}_01"
result = fit_flowers(model, data, OUTPUT_DIR, epochs=EPOCHS,
    batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, seed=SEED, device=device)
print(result["metrics"])


## 5. Графики и примеры
Функция обучения возвращает веса лучшей эпохи по macro-F1 валидации. Все эпохи сохранены. Рассмотрите loss и качество отдельно; метрики фазы не накапливаются за предыдущие эпохи.

In [ ]:
fig = plot_learning(result)
fig.savefig(OUTPUT_DIR / "curves.pdf", bbox_inches="tight")
display(pd.read_csv(OUTPUT_DIR / "validation_report.csv", index_col=0))
display(pd.read_csv(OUTPUT_DIR / "confusion_matrix.csv", index_col=0))


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig_cm, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(result["validation"]["actual"],
    result["validation"]["predicted"], labels=range(5),
    display_labels=data["classes"], normalize="true", ax=ax)
fig_cm.tight_layout()
fig_cm.savefig(OUTPUT_DIR / "confusion.pdf", bbox_inches="tight")


In [ ]:
model = result["model"]
model.eval()
fig_samples, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax in axes.flat:
    ax.axis("off")
for ax, index in zip(axes.flat, data["validation_indices"][:9]):
    image, label = data["validation_dataset"][int(index)]
    with torch.no_grad():
        predicted = int(model(image.unsqueeze(0).to(device)).argmax(1).item())
    ax.imshow(denormalize(image).permute(1, 2, 0))
    ax.set_title(f"True: {data['classes'][label]}\nPred: {data['classes'][predicted]}")
fig_samples.tight_layout()
fig_samples.savefig(OUTPUT_DIR / "examples.pdf", bbox_inches="tight")


## 6. Сравнение трёх архитектур
После выполнения трёх запусков соберите таблицу. Укажите внешнее предобучение, число параметров, эпоху, метрики и время. Добавьте анализ ошибок и отдельную диаграмму времени. Лучшая валидация не является независимым тестом.

In [ ]:
records = []
for architecture in ("original", "transfer", "student"):
    folder = Path("runs") / f"lab3_{architecture}_01"
    if not (folder / "manifest.json").is_file():
        print("Run is still required:", architecture)
        continue
    manifest = json.loads((folder / "manifest.json").read_text())
    records.append({"model": architecture, **manifest["parameters"],
        **manifest["validation_metrics"], "best_epoch": manifest["best_epoch"],
        "seconds_fit_and_checkpoint_io": manifest["seconds_fit_and_checkpoint_io"],
        "pretrained": manifest["architecture"].get("pretrained")})
comparison = pd.DataFrame(records)
display(comparison)
if len(comparison) == 3:
    comparison.to_csv("runs/lab3_comparison.csv", index=False)
    comparison.plot.bar(x="model", y="macro_f1", legend=False)
    plt.ylabel("Validation macro-F1")
    plt.tight_layout()
else:
    print("The three-model comparison is incomplete.")


## Вывод и защита
Опишите устройство трёх моделей, собственное изменение и его эффект, худшие классы, роль предобучения, режим BatchNorm, потери и метрики. Ответьте на контрольные вопросы методички.

**Самопроверка:** единое разбиение; пять логитов; CrossEntropyLoss без предварительного Softmax; backbone действительно заморожен; цвета восстановлены; собственная модель изменена и объяснена; все три опыта представлены.